# YOLOv5 Instance Segmentation Training - KTP Segmentation

Notebook ini untuk training model **YOLOv5 Instance Segmentation** menggunakan dataset dari Roboflow.

**Kompatibel dengan:** Google Colab & Kaggle

**Flow:** Download Dataset (Roboflow) → Setup YOLOv5 → Training Segmentation → Export Model

> **Penting:** Dataset harus berupa **Instance Segmentation** dengan anotasi **polygon** di Roboflow. Export format: `yolov5`

## 1. Cek Environment (Colab / Kaggle)

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    PLATFORM = 'colab'
    print('✅ Running on Google Colab')
except:
    IN_COLAB = False
    try:
        import kaggle
        PLATFORM = 'kaggle'
        print('✅ Running on Kaggle')
    except:
        PLATFORM = 'local'
        print('✅ Running locally')

# Path dasar
BASE_DIR = '/content' if IN_COLAB else '/kaggle/working' if PLATFORM == 'kaggle' else '.'
print(f'Base directory: {BASE_DIR}')

## 2. Install Dependencies

In [ ]:
# Install Roboflow & YOLOv5 dependencies
!pip install -q roboflow
!pip install -q torch torchvision  # YOLOv5 membutuhkan PyTorch

## 3. Clone YOLOv5 Repository

In [ ]:
import os

YOLOV5_DIR = os.path.join(BASE_DIR, 'yolov5')

if not os.path.exists(YOLOV5_DIR):
    !git clone https://github.com/ultralytics/yolov5 {YOLOV5_DIR}
else:
    print('YOLOv5 sudah ada')

%cd {YOLOV5_DIR}
!pip install -q -r requirements.txt

## 4. Download Dataset dari Roboflow

In [ ]:
from roboflow import Roboflow

# Ganti API_AUTH_KEY dengan API key Anda dari https://app.roboflow.com/settings/api
API_KEY = "API_AUTH_KEY"  # Paste API key Roboflow Anda di sini

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("ocr-lw2bp").project("ktp-swgmentation-wkrdr")
version = project.version(1)

# Format "yolov5" - pastikan project Roboflow bertipe Instance Segmentation dengan polygon
dataset = version.download("yolov5")

# dataset bisa berupa string path atau object
DATASET_PATH = dataset if isinstance(dataset, str) else dataset.location
print(f'\n✅ Dataset downloaded to: {DATASET_PATH}')

## 5. Konfigurasi Training

In [ ]:
import os

# Path dataset (dari cell download, atau fallback)
DATASET_PATH = globals().get('DATASET_PATH', os.path.join(BASE_DIR, 'ktp-swgmentation-wkrdr-1'))
DATA_YAML = os.path.join(DATASET_PATH, 'data.yaml')

# Pastikan data.yaml ada
if not os.path.exists(DATA_YAML):
    # Coba cari di subfolder
    for root, dirs, files in os.walk(BASE_DIR):
        if 'data.yaml' in files:
            DATA_YAML = os.path.join(root, 'data.yaml')
            DATASET_PATH = root
            break

print(f'Dataset path: {DATASET_PATH}')
print(f'Data YAML: {DATA_YAML}')
print(f'Exists: {os.path.exists(DATA_YAML)}')

In [ ]:
# Baca data.yaml untuk cek kelas
if os.path.exists(DATA_YAML):
    with open(DATA_YAML, 'r') as f:
        print(f.read())

## 6. Jalankan Training

In [ ]:
# Parameter training Instance Segmentation
EPOCHS = 50          # Jumlah epoch (naikkan untuk hasil lebih baik)
BATCH_SIZE = 16      # Batch size (turunkankan jika OOM)
IMG_SIZE = 640       # Ukuran gambar input
# Model SEGMENTATION: yolov5n-seg, yolov5s-seg, yolov5m-seg, yolov5l-seg, yolov5x-seg
MODEL = 'yolov5s-seg'
PROJECT = 'runs/ktp_segmentation'
NAME = 'train'

In [ ]:
# Instance Segmentation: gunakan segment/train.py
!python segment/train.py \
    --img {IMG_SIZE} \
    --batch {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --data {DATA_YAML} \
    --weights {MODEL}.pt \
    --project {PROJECT} \
    --name {NAME} \
    --exist-ok

## 6.1 Validasi & Inference (Opsional)

## 6.2 Inference pada Gambar Test (Opsional)

In [ ]:
# Validasi model segmentation (jalankan setelah training selesai)
best_weights = os.path.join(YOLOV5_DIR, PROJECT, NAME, 'weights', 'best.pt')
if os.path.exists(best_weights):
    !python segment/val.py --weights {best_weights} --data {DATA_YAML} --img {IMG_SIZE}
else:
    print('Jalankan training terlebih dahulu.')

In [ ]:
# Inference segmentation pada gambar test
import glob
from IPython.display import Image, display
best_weights = os.path.join(YOLOV5_DIR, PROJECT, NAME, 'weights', 'best.pt')
test_imgs = glob.glob(os.path.join(DATASET_PATH, 'test', 'images', '*.*'))
if os.path.exists(best_weights) and test_imgs:
    test_img = test_imgs[0]
    !python segment/predict.py --weights {best_weights} --source {test_img} --img {IMG_SIZE}
    # Tampilkan hasil (output di runs/predict-seg/)
    pred_dir = os.path.join(YOLOV5_DIR, 'runs', 'predict-seg')
    if os.path.exists(pred_dir):
        pred_imgs = glob.glob(os.path.join(pred_dir, 'exp*', '*.jpg')) + glob.glob(os.path.join(pred_dir, 'exp*', '*.png'))
        if pred_imgs:
            display(Image(filename=sorted(pred_imgs)[-1], width=600))
else:
    print('Jalankan training terlebih dahulu atau cek path dataset.')

In [ ]:
# Validasi model segmentation (jalankan setelah training selesai)
best_weights = os.path.join(YOLOV5_DIR, PROJECT, NAME, 'weights', 'best.pt')
if os.path.exists(best_weights):
    !python segment/val.py --weights {best_weights} --data {DATA_YAML} --img {IMG_SIZE}
else:
    print('Jalankan training terlebih dahulu.')

## 7. Validasi & Hasil

In [ ]:
from IPython.display import Image, display

# Tampilkan hasil training (curves)
results_dir = os.path.join(YOLOV5_DIR, PROJECT, NAME)
results_img = os.path.join(results_dir, 'results.png')

if os.path.exists(results_img):
    display(Image(filename=results_img, width=800))
else:
    print('Hasil training belum tersedia. Jalankan cell training terlebih dahulu.')

In [ ]:
# Path model terbaik
BEST_WEIGHTS = os.path.join(results_dir, 'weights', 'best.pt')
LAST_WEIGHTS = os.path.join(results_dir, 'weights', 'last.pt')

if os.path.exists(BEST_WEIGHTS):
    print(f'✅ Model terbaik: {BEST_WEIGHTS}')
    print(f'✅ Model terakhir: {LAST_WEIGHTS}')
else:
    print('Model belum tersedia. Pastikan training selesai.')

## 8. Export Model (Opsional)

In [ ]:
# Export ke format ONNX (untuk deployment)
if os.path.exists(BEST_WEIGHTS):
    !python export.py --weights {BEST_WEIGHTS} --include onnx
    print('\n✅ Model exported ke ONNX')
else:
    print('Jalankan training terlebih dahulu.')

## 9. Download Model (Colab)

In [ ]:
# Di Colab: download model ke komputer lokal
if IN_COLAB and os.path.exists(BEST_WEIGHTS):
    from google.colab import files
    files.download(BEST_WEIGHTS)
    print('✅ Download best.pt dimulai')
elif PLATFORM == 'kaggle':
    print('Di Kaggle: model tersimpan di Output. Cek tab Output setelah run selesai.')
else:
    print(f'Model tersimpan di: {BEST_WEIGHTS}')

## Catatan

- **Colab GPU**: Aktifkan GPU di Runtime → Change runtime type → GPU (T4 gratis)
- **Kaggle GPU**: Pilih GPU di Settings → Accelerator
- **API Key**: Dapatkan di [Roboflow Settings](https://app.roboflow.com/settings/api)
- **OOM**: Jika Out of Memory, turunkan `BATCH_SIZE` (misal 8 atau 4)
- **Model size**: `yolov5n-seg` paling cepat, `yolov5x-seg` paling akurat tapi lambat
- **Instance Segmentation**: Gunakan `segment/train.py`, weights `*-seg.pt`, dataset dengan polygon annotations